In [0]:
%sql
-- ===============================================================================
-- Creating the Gold layer for Streamlit web app deployment
-- ===============================================================================

CREATE SCHEMA IF NOT EXISTS pvdaq_catalog.gold;

USE CATALOG pvdaq_catalog;
USE SCHEMA gold;
SELECT current_catalog(), current_schema();


In [0]:
%pip install pvanalytics
dbutils.library.restartPython()

In [0]:
# =============================================================================
# Gold layer: precompute daily and annual performance metrics per system.
#
# Paste this into a cell in the "PVDAQ Data - Gold" notebook and run it once.
# It moves the pvanalytics math out of the web app and into the pipeline, so
# the Streamlit app only reads a few thousand rows instead of recomputing
# performance ratios from ~2.5M rows of raw sensor data on every page load.
#
# Rerun this cell whenever the Silver layer changes.
# =============================================================================

import pandas as pd
from pvanalytics.metrics import performance_ratio_nrel

spark.sql("CREATE SCHEMA IF NOT EXISTS pvdaq_catalog.gold")

DAYTIME_POA_THRESHOLD = 50  # W/m^2, same cutoff used in the Silver notebook

# Load the full joined time series and the system metadata once
df = (
    spark.table("pvdaq_catalog.silver.pvdata_2020_joined")
    .toPandas()
)
df["utc_measured_on"] = pd.to_datetime(df["utc_measured_on"])
df = df.set_index("utc_measured_on").sort_index()

systems = (
    spark.table("pvdaq_catalog.silver.system")
    .select("system_id", "power", "public_name")
    .toPandas()
)
systems["power"] = pd.to_numeric(systems["power"], errors="coerce")

daily_rows = []
annual_rows = []

for system_id, sys_df in df.groupby("system_id"):

    meta = systems[systems["system_id"] == system_id]
    if meta.empty or pd.isna(meta["power"].iloc[0]):
        print(f"Skipping system {system_id}: no DC capacity in silver.system")
        continue

    # Same normalization as plot_system_pr_and_power(): values above 10,000 are
    # assumed to be watts and converted to kW
    raw_power = float(meta["power"].iloc[0])
    pdc0 = raw_power / 1000.0 if raw_power > 10000 else raw_power

    daytime_full = sys_df[sys_df["poa_irradiance"] >= DAYTIME_POA_THRESHOLD]
    if daytime_full.empty:
        print(f"Skipping system {system_id}: no daytime records")
        continue

    pr_annual = performance_ratio_nrel(
        poa_global=daytime_full["poa_irradiance"],
        temp_air=daytime_full["ambient_temp"],
        wind_speed=daytime_full["wind_speed"],
        pac=daytime_full["ac_power_kw"],
        pdc0=pdc0,
    )

    annual_rows.append(
        {
            "system_id": int(system_id),
            "public_name": str(meta["public_name"].iloc[0]),
            "pdc0_kw": float(pdc0),
            "pr_annual": float(pr_annual),
        }
    )

    for date, day_df in sys_df.groupby(sys_df.index.date):
        daytime_data = day_df[day_df["poa_irradiance"] >= DAYTIME_POA_THRESHOLD]
        if daytime_data.empty:
            continue

        pr = performance_ratio_nrel(
            poa_global=daytime_data["poa_irradiance"],
            temp_air=daytime_data["ambient_temp"],
            wind_speed=daytime_data["wind_speed"],
            pac=daytime_data["ac_power_kw"],
            pdc0=pdc0,
        )

        # Mean over the whole day, including night, matching the notebook
        daily_rows.append(
            {
                "system_id": int(system_id),
                "date": date,
                "pr": float(pr),
                "avg_ac_power_kw": float(day_df["ac_power_kw"].mean()),
            }
        )

    print(f"System {system_id}: annual PR {pr_annual:.3f}, pdc0 {pdc0:.1f} kW")

# -----------------------------------------------------------------------------
# Write the Gold tables
# -----------------------------------------------------------------------------

daily_pdf = pd.DataFrame(daily_rows)
daily_pdf["date"] = pd.to_datetime(daily_pdf["date"])

annual_pdf = pd.DataFrame(annual_rows)

(
    spark.createDataFrame(daily_pdf)
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("pvdaq_catalog.gold.system_daily_performance")
)

(
    spark.createDataFrame(annual_pdf)
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("pvdaq_catalog.gold.system_annual_performance")
)

print(f"\nWrote {len(daily_pdf):,} daily rows and {len(annual_pdf)} annual rows.")

In [0]:
# =============================================================================
# Gold layer: export the performance tables as flat files for the public app.
#
# Add this as the LAST cell of the "PVDAQ Data - Gold" notebook, after the cell
# that writes system_daily_performance and system_annual_performance.
#
# It writes CSVs directly into the Git folder in your workspace, so the files
# show up as uncommitted changes you can commit and push from the Databricks
# Git UI. Streamlit Community Cloud then deploys them straight from GitHub.
#
# The exported data is a few hundred KB. No credentials, no warehouse, no
# Databricks account needed by anyone viewing the site.
# =============================================================================

import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

# -----------------------------------------------------------------------------
# Point this at the data/ directory inside your Git folder.
# Confirm the username segment matches your workspace path.
# -----------------------------------------------------------------------------
EXPORT_DIR = Path(
    "/Workspace/Users/ryanmasson@gmail.com/solar-performance/streamlit_app_public/data"
)

EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# Read back what the pipeline just wrote
# -----------------------------------------------------------------------------

daily = (
    spark.table("pvdaq_catalog.gold.system_daily_performance")
    .orderBy("system_id", "date")
    .toPandas()
)

annual = (
    spark.table("pvdaq_catalog.gold.system_annual_performance")
    .orderBy("system_id")
    .toPandas()
)

# Dates serialize as YYYY-MM-DD rather than a full timestamp
daily["date"] = pd.to_datetime(daily["date"]).dt.strftime("%Y-%m-%d")

# -----------------------------------------------------------------------------
# Write
# -----------------------------------------------------------------------------

daily_path = EXPORT_DIR / "system_daily_performance.csv"
annual_path = EXPORT_DIR / "system_annual_performance.csv"

daily.to_csv(daily_path, index=False)
annual.to_csv(annual_path, index=False)

manifest = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "source": "NREL PVDAQ / DOE Open Energy Data Initiative, 2020",
    "systems": int(annual["system_id"].nunique()),
    "daily_rows": int(len(daily)),
    "daytime_poa_threshold_w_m2": 50,
}

with open(EXPORT_DIR / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print(f"Wrote {len(daily):,} daily rows -> {daily_path}")
print(f"Wrote {len(annual):,} annual rows -> {annual_path}")
print(f"\nTotal size: "
      f"{(daily_path.stat().st_size + annual_path.stat().st_size) / 1024:.0f} KB")
print("\nNext: commit and push these files from the Git folder UI.")

# -----------------------------------------------------------------------------
# If writing to /Workspace fails on your runtime, write to a UC volume instead
# and download the files from the Catalog UI:
#
#   VOLUME_DIR = "/Volumes/pvdaq_catalog/gold/exports"
#   spark.sql("CREATE VOLUME IF NOT EXISTS pvdaq_catalog.gold.exports")
#   daily.to_csv(f"{VOLUME_DIR}/system_daily_performance.csv", index=False)
# -----------------------------------------------------------------------------